# Laboratório — LGN, TCL e Monte Carlo

O laboratório separa estabilização, forma do erro e cálculo por simulação. Usaremos distribuições Bernoulli, exponencial, lognormal e Cauchy; depois estimaremos $\pi$, uma integral e o custo de minibatches e eventos raros.

**Dependências:** Python ≥ 3.10, NumPy ≥ 1.26, SciPy ≥ 1.11 e Matplotlib ≥ 3.8.  
**Seed principal:** `20260907`.


## 1. Ambiente

Criamos geradores locais, evitando estado aleatório global. Seeds derivadas separam experimentos sem impedir sua reprodução.


In [1]:
import sys
import math
import numpy as np
import scipy
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import norm, gamma, skew
from scipy.special import erf

SEED = 20260907
rng = np.random.default_rng(SEED)

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__} | SciPy: {scipy.__version__} | Matplotlib: {matplotlib.__version__}")
print(f"Seed: {SEED}")

Python: 3.12.13
NumPy: 2.3.5 | SciPy: 1.17.0 | Matplotlib: 3.10.8
Seed: 20260907


## 2. LGN em uma frequência Bernoulli

Para $X_i\sim\mathrm{Bernoulli}(0,3)$, a média acumulada é a frequência de sucessos. Ela oscila no início e tende a se concentrar perto de 0,3.


In [2]:
p = 0.3
n_max = 200_000
bernoulli = rng.binomial(1, p, size=n_max)
media_acumulada = np.cumsum(bernoulli) / np.arange(1, n_max + 1)

for n in [10, 100, 1_000, 10_000, 200_000]:
    print(f"n={n:>6}: frequência={media_acumulada[n-1]:.6f}")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(np.arange(1, n_max + 1), media_acumulada, color="#2563eb", linewidth=1)
ax.axhline(p, color="#dc2626", linestyle="--", label="p=0,3")
ax.set_xscale("log")
ax.set(xlabel="Número de tentativas (escala log)", ylabel="Frequência acumulada", title="LGN: estabilização da frequência relativa")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

assert abs(media_acumulada[-1] - p) < 0.003

n=    10: frequência=0.300000
n=   100: frequência=0.290000
n=  1000: frequência=0.313000
n= 10000: frequência=0.303800
n=200000: frequência=0.299145


## 3. Chebyshev: limite geral e conservador

Para médias de exponenciais com $\mu=1$, $\sigma^2=1$, $n=100$ e $\varepsilon=0,2$, Chebyshev fornece o limite $1/(100\cdot0,2^2)=0,25$. Simulamos a probabilidade real para comparar.


In [3]:
rng_cheb = np.random.default_rng(SEED + 1)
n_rep, n, epsilon = 80_000, 100, 0.2
medias_exp = rng_cheb.exponential(scale=1.0, size=(n_rep, n)).mean(axis=1)
p_emp = np.mean(np.abs(medias_exp - 1.0) >= epsilon)
limite_cheb = 1.0 / (n * epsilon**2)

print(f"Probabilidade empírica: {p_emp:.6f}")
print(f"Limite de Chebyshev: {limite_cheb:.6f}")

assert p_emp < limite_cheb
assert np.isclose(limite_cheb, 0.25)

Probabilidade empírica: 0.044525
Limite de Chebyshev: 0.250000


## 4. TCL com dados exponenciais

As observações originais são assimétricas. Repetimos a amostragem e padronizamos as médias para $n=1,5,30,100$. A forma se aproxima da normal padrão à medida que $n$ cresce.


In [4]:
rng_clt = np.random.default_rng(SEED + 2)
n_rep_clt = 40_000
tamanhos = [1, 5, 30, 100]
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
grade_z = np.linspace(-4, 5, 500)
assimetrias = {}

for ax, tamanho in zip(axes.flat, tamanhos):
    medias = rng_clt.exponential(size=(n_rep_clt, tamanho)).mean(axis=1)
    z_medias = np.sqrt(tamanho) * (medias - 1.0)
    assimetrias[tamanho] = skew(z_medias, bias=False)
    ax.hist(z_medias, bins=70, density=True, alpha=0.55, color="#2563eb")
    ax.plot(grade_z, norm.pdf(grade_z), color="#dc2626", linewidth=2)
    ax.set_title(f"n={tamanho}; assimetria={assimetrias[tamanho]:.3f}")
    ax.grid(alpha=0.2)

fig.supxlabel("Média padronizada")
fig.supylabel("Densidade")
fig.suptitle("TCL: médias de exponenciais × normal padrão", y=1.02)
plt.tight_layout()
plt.show()

print("Assimetrias:", {k: round(v, 4) for k, v in assimetrias.items()})
assert assimetrias[1] > 1.8
assert assimetrias[100] < 0.25

Assimetrias: {1: np.float64(1.957), 5: np.float64(0.8854), 30: np.float64(0.3428), 100: np.float64(0.1943)}


## 5. Aproximação TCL versus resultado exato

Para a média de 40 exponenciais de taxa 1, $40\bar X$ segue uma gama com forma 40 e escala 1. Comparamos $P(\bar X>1,2)$ pelo TCL e pela distribuição exata.


In [5]:
n_exp = 40
limite = 1.2
z_limite = (limite - 1.0) / (1.0 / np.sqrt(n_exp))
p_tcl = norm.sf(z_limite)
p_exata = gamma.sf(limite, a=n_exp, scale=1/n_exp)
erro_abs = abs(p_tcl - p_exata)

print(f"z: {z_limite:.9f}")
print(f"TCL: {p_tcl:.9f} | exata: {p_exata:.9f} | erro absoluto: {erro_abs:.9f}")

assert np.isclose(p_tcl, 0.1029516054, atol=1e-10)
assert erro_abs < 0.01

z: 1.264911064
TCL: 0.102951605 | exata: 0.107276466 | erro absoluto: 0.004324861


## 6. Não existe um tamanho universal

Comparamos médias de lognormais moderadas e muito assimétricas. Mesmo para o mesmo $n$, a aproximação é mais lenta quando a cauda é mais pesada.


In [6]:
rng_heavy = np.random.default_rng(SEED + 3)
n_rep_heavy = 30_000
resultado_skew = {}
for sigma_log in [0.5, 1.5]:
    media_pop = np.exp(sigma_log**2 / 2)
    var_pop = (np.exp(sigma_log**2)-1) * np.exp(sigma_log**2)
    for tamanho in [30, 300]:
        medias = rng_heavy.lognormal(0, sigma_log, size=(n_rep_heavy, tamanho)).mean(axis=1)
        z = np.sqrt(tamanho) * (medias - media_pop) / np.sqrt(var_pop)
        resultado_skew[(sigma_log, tamanho)] = skew(z, bias=False)

for chave, valor in resultado_skew.items():
    print(f"sigma_log={chave[0]}, n={chave[1]} -> assimetria das médias={valor:.4f}")

assert resultado_skew[(1.5, 30)] > resultado_skew[(0.5, 30)]
assert resultado_skew[(1.5, 30)] > 1.0

sigma_log=0.5, n=30 -> assimetria das médias=0.3155
sigma_log=0.5, n=300 -> assimetria das médias=0.1178
sigma_log=1.5, n=30 -> assimetria das médias=3.4125
sigma_log=1.5, n=300 -> assimetria das médias=2.0433


## 7. Cauchy: quando a média não estabiliza

A Cauchy padrão não possui média finita. Trajetórias da média acumulada podem sofrer saltos tardios, enquanto a mediana acumulada é mais estável. O gráfico é um contraexemplo, não uma prova do teorema.


In [7]:
rng_cauchy = np.random.default_rng(SEED + 4)
n_cauchy = 30_000
cauchy = rng_cauchy.standard_cauchy(n_cauchy)
media_cauchy = np.cumsum(cauchy) / np.arange(1, n_cauchy + 1)
pontos = np.unique(np.geomspace(10, n_cauchy, 220).astype(int))
mediana_cauchy = np.array([np.median(cauchy[:i]) for i in pontos])

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(np.arange(1, n_cauchy + 1), media_cauchy, alpha=0.75, label="Média acumulada")
ax.plot(pontos, mediana_cauchy, linewidth=2, label="Mediana acumulada")
ax.axhline(0, color="#111827", linestyle="--")
ax.set_xscale("log")
ax.set_ylim(-8, 8)
ax.set(xlabel="n (escala log)", ylabel="Estimativa", title="Cauchy: a LGN clássica da média não se aplica")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

print(f"Média final: {media_cauchy[-1]:.6f} | mediana final: {np.median(cauchy):.6f}")
assert abs(np.median(cauchy)) < 0.05

Média final: -0.323124 | mediana final: -0.007879


## 8. Monte Carlo para estimar $\pi$

O indicador de um ponto cair no círculo tem média $\pi/4$. Usamos 500 mil pontos, calculamos o erro-padrão e verificamos se $\pi$ está em uma faixa aproximada de 1,96 erros-padrão.


In [8]:
rng_pi = np.random.default_rng(SEED + 5)
n_pi = 500_000
pontos_pi = rng_pi.uniform(-1, 1, size=(n_pi, 2))
indicador = np.sum(pontos_pi**2, axis=1) <= 1
pi_hat = 4 * indicador.mean()
se_pi = 4 * indicador.std(ddof=1) / np.sqrt(n_pi)

print(f"pi estimado: {pi_hat:.9f}")
print(f"erro-padrão MC: {se_pi:.9f}")
print(f"erro observado: {pi_hat - np.pi:+.9f}")

assert abs(pi_hat - np.pi) < 3 * se_pi
assert 0.002 < se_pi < 0.003

pi estimado: 3.136896000
erro-padrão MC: 0.002327004
erro observado: -0.004696654


## 9. A taxa $1/\sqrt n$

Repetimos a estimativa de $\pi$ para quatro tamanhos. O RMSE entre repetições deve cair aproximadamente pela metade quando $n$ quadruplica.


In [9]:
rng_rate = np.random.default_rng(SEED + 6)
tamanhos_mc = np.array([200, 800, 3_200, 12_800])
n_repeticoes = 1_500
rmses = []

for tamanho in tamanhos_mc:
    pts = rng_rate.uniform(-1, 1, size=(n_repeticoes, tamanho, 2))
    estimativas = 4 * (np.sum(pts**2, axis=2) <= 1).mean(axis=1)
    rmse = np.sqrt(np.mean((estimativas - np.pi)**2))
    rmses.append(rmse)
    print(f"n={tamanho:>5}: RMSE={rmse:.6f}")

rmses = np.array(rmses)
razoes = rmses[1:] / rmses[:-1]
print("Razões ao quadruplicar n:", np.round(razoes, 4))
assert np.all((razoes > 0.43) & (razoes < 0.57))

n=  200: RMSE=0.117588
n=  800: RMSE=0.058267
n= 3200: RMSE=0.028876
n=12800: RMSE=0.014533
Razões ao quadruplicar n: [0.4955 0.4956 0.5033]


## 10. Uma integral como esperança

Se $U\sim\mathrm{Uniforme}(0,1)$, então $E[e^{-U^2}]=\int_0^1e^{-x^2}dx$. O valor exato é $\sqrt\pi\,\mathrm{erf}(1)/2$.


In [10]:
rng_integral = np.random.default_rng(SEED + 7)
n_integral = 700_000
u = rng_integral.random(n_integral)
valores = np.exp(-u**2)
integral_hat = valores.mean()
se_integral = valores.std(ddof=1) / np.sqrt(n_integral)
integral_exata = np.sqrt(np.pi) * erf(1) / 2

print(f"Estimativa: {integral_hat:.9f}")
print(f"Exata: {integral_exata:.9f}")
print(f"Erro-padrão: {se_integral:.9f}")

assert np.isclose(integral_exata, 0.7468241328, atol=1e-10)
assert abs(integral_hat - integral_exata) < 3 * se_integral

Estimativa: 0.746552632
Exata: 0.746824133
Erro-padrão: 0.000240362


## 11. Processamento em lotes sem mudar a sequência

Gerar pares em blocos reduz memória. Com a mesma seed e a mesma ordem de consumo do gerador, o resultado coincide exatamente com a geração integral.


In [11]:
def estima_pi_em_lotes(n_total, lote, seed):
    gerador = np.random.default_rng(seed)
    dentro_total = 0
    processados = 0
    while processados < n_total:
        atual = min(lote, n_total - processados)
        pontos = gerador.uniform(-1, 1, size=(atual, 2))
        dentro_total += np.count_nonzero(np.sum(pontos**2, axis=1) <= 1)
        processados += atual
    return 4 * dentro_total / n_total

n_lotes = 123_457
seed_lotes = SEED + 8
pi_lotes = estima_pi_em_lotes(n_lotes, lote=10_000, seed=seed_lotes)
gerador_unico = np.random.default_rng(seed_lotes)
pontos_unicos = gerador_unico.uniform(-1, 1, size=(n_lotes, 2))
pi_unico = 4 * np.mean(np.sum(pontos_unicos**2, axis=1) <= 1)

print(f"Em lotes: {pi_lotes:.9f} | integral: {pi_unico:.9f}")
assert pi_lotes == pi_unico

Em lotes: 3.144414654 | integral: 3.144414654


## 12. Variância de médias de minibatch

Se gradientes individuais idealizados têm variância 4 e são independentes, a variância da média de um lote de tamanho $b$ é $4/b$. A simulação mostra o retorno de raiz quadrada.


In [12]:
rng_batch = np.random.default_rng(SEED + 9)
n_batches = 20_000
for b in [1, 4, 16, 64, 256]:
    medias_batch = rng_batch.normal(loc=0, scale=2, size=(n_batches, b)).mean(axis=1)
    var_emp = medias_batch.var(ddof=1)
    var_teorica = 4 / b
    print(f"b={b:>3}: variância empírica={var_emp:.6f} | teórica={var_teorica:.6f}")
    assert abs(var_emp / var_teorica - 1) < 0.04

b=  1: variância empírica=4.056824 | teórica=4.000000
b=  4: variância empírica=1.015597 | teórica=1.000000
b= 16: variância empírica=0.249389 | teórica=0.250000
b= 64: variância empírica=0.062116 | teórica=0.062500
b=256: variância empírica=0.015801 | teórica=0.015625


## 13. Evento raro e erro relativo

Estimamos $P(Z>4)$ com 100 mil normais. O número esperado de ocorrências é pequeno; por isso o erro relativo teórico é alto mesmo quando o erro absoluto parece pequeno.


In [13]:
rng_rare = np.random.default_rng(SEED + 10)
n_rare = 100_000
p_rare = norm.sf(4)
z_rare = rng_rare.standard_normal(n_rare)
sucessos = np.count_nonzero(z_rare > 4)
p_hat_rare = sucessos / n_rare
rel_se_teorico = np.sqrt(p_rare * (1-p_rare) / n_rare) / p_rare

print(f"p teórico: {p_rare:.10f}")
print(f"sucessos observados: {sucessos} | esperados: {n_rare*p_rare:.3f}")
print(f"estimativa: {p_hat_rare:.10f} | erro-padrão relativo teórico: {rel_se_teorico:.3%}")

assert np.isclose(p_rare, 3.1671241833e-5, rtol=1e-10)
assert 0.5 < rel_se_teorico < 0.6

p teórico: 0.0000316712
sucessos observados: 0 | esperados: 3.167
estimativa: 0.0000000000 | erro-padrão relativo teórico: 56.190%


## 14. Reprodutibilidade e replicação

A mesma seed reproduz a mesma sequência. Uma seed diferente gera uma replicação independente. Validade depende do conjunto de replicações e das hipóteses, não de escolher uma sequência favorável.


In [14]:
a = np.random.default_rng(12345).random(8)
b = np.random.default_rng(12345).random(8)
c = np.random.default_rng(12346).random(8)

print("Mesma seed, vetores iguais:", np.array_equal(a, b))
print("Seed diferente, vetores iguais:", np.array_equal(a, c))
assert np.array_equal(a, b)
assert not np.array_equal(a, c)

Mesma seed, vetores iguais: True
Seed diferente, vetores iguais: False


## 15. Checagens finais

As asserções confirmam as relações essenciais: variância da média, aproximação TCL, identidade da integral, taxa de Monte Carlo e reprodutibilidade.


In [15]:
assert np.isclose(1 / np.sqrt(100), 0.1)
assert p_emp < limite_cheb
assert erro_abs < 0.01
assert abs(pi_hat - np.pi) < 3 * se_pi
assert abs(integral_hat - integral_exata) < 3 * se_integral
assert np.all((razoes > 0.43) & (razoes < 0.57))

print("Todas as verificações numéricas foram concluídas com sucesso.")

Todas as verificações numéricas foram concluídas com sucesso.


## Conclusões

- A frequência Bernoulli estabilizou, ilustrando a LGN.
- Chebyshev forneceu um limite válido, porém conservador.
- Médias de exponenciais ficaram progressivamente mais normais, como prevê o TCL.
- O erro da aproximação TCL foi medido contra uma probabilidade gama exata.
- A Cauchy mostrou por que momentos finitos são hipóteses, não detalhes.
- Monte Carlo estimou $\pi$ e uma integral com erro-padrão explícito.
- Ao quadruplicar $n$, o RMSE caiu aproximadamente pela metade.
- Processamento em lotes preservou o estimador e controlou memória.
- Minibatches exibiram variância inversamente proporcional ao tamanho do lote.
- O evento raro apresentou erro relativo alto com poucos sucessos.

Próximo passo: estatística descritiva, robustez e outliers na Aula 11.
